# ISIC 2019 Bronze ingestion

This notebook expects the ISIC 2019 Bronze tables/folders to already exist and the source archives to already be uploaded to landing.

Run `05_create_bronze_tables_and_folders.ipynb` before this notebook.


In [ ]:
%run ../_setup_env


In [ ]:
from datetime import datetime, timezone
from pathlib import Path
from uuid import uuid4

from pyspark.sql import functions as F

from data_platform.files import IMAGE_SUFFIXES, list_files_by_suffix, materialize_zip_images_and_metadata, sha256_file
from data_platform.setup import build_layout, load_dataset_config, load_yaml_config

DATASET_CONFIG = load_dataset_config(CONFIG_ROOT / "datasets" / "isic_2019.yaml")
STORAGE_CONFIG = load_yaml_config(CONFIG_ROOT / "storage.yaml")
DATASET_CONFIG["landing_root"] = STORAGE_CONFIG["landing_root"]
DATASET_CONFIG["storage_root"] = STORAGE_CONFIG["storage_root"]
layout = build_layout(DATASET_CONFIG)
landing_paths = layout["landing_paths"]
bronze_paths = layout["bronze_paths"]
bronze_tables = layout["bronze_tables"]

ARCHIVES = [
    {
        "source_split": "train",
        "archive_filename": "ISIC-2019-train-images.zip",
        "metadata_filename": "metadata.csv", # expected metadata filename inside archive
    },
    {
        "source_split": "test",
        "archive_filename": "ISIC-2019-test-images.zip",
        "metadata_filename": "metadata.csv", # expected metadata filename inside archive
    },
]
OVERWRITE_BRONZE_FILES = False

for archive in ARCHIVES:
    source_split = archive["source_split"]
    archive["archive_dbfs_path"] = f"{landing_paths['dataset_archive'].rstrip('/')}/{archive['archive_filename']}"
    archive["archive_local_path"] = Path("/dbfs") / archive["archive_dbfs_path"].removeprefix("dbfs:/")
    archive["bronze_images_path"] = f"{bronze_paths['images'].rstrip('/')}/{source_split}"
    archive["bronze_images_local_root"] = Path("/dbfs") / archive["bronze_images_path"].removeprefix("dbfs:/")
    archive["metadata_dir_path"] = f"{bronze_paths['metadata'].rstrip('/')}/{source_split}"
    archive["metadata_local_root"] = Path("/dbfs") / archive["metadata_dir_path"].removeprefix("dbfs:/")
    archive["metadata_target_path"] = f"{archive['metadata_dir_path'].rstrip('/')}/{archive['metadata_filename']}"

for archive in ARCHIVES:
    print(f"{archive['source_split']} archive landing path: {archive['archive_dbfs_path']}")
    print(f"{archive['source_split']} Bronze images path: {archive['bronze_images_path']}")
    print(f"{archive['source_split']} Bronze metadata path: {archive['metadata_target_path']}")


## Preflight

Verify that the required train and test archives already exist in landing before doing any extraction or table writes.


In [ ]:
for archive in ARCHIVES:
    if not archive["archive_local_path"].exists():
        raise FileNotFoundError(f"Missing archive: {archive['archive_dbfs_path']}")

print("All required source archives are present.")


## Materialize Bronze Files

This step extracts each landing archive directly into its Bronze images split path and copies each embedded metadata CSV into Bronze metadata.

In [ ]:
for archive in ARCHIVES:
    materialized = materialize_zip_images_and_metadata(
        archive_path=archive["archive_local_path"],
        image_root=archive["bronze_images_local_root"],
        metadata_root=archive["metadata_local_root"],
        metadata_filename=archive["metadata_filename"],
        overwrite=OVERWRITE_BRONZE_FILES,
    )
    print(
        f"{materialized['image_status'].capitalize()} Bronze images: "
        f"{archive['bronze_images_path']} ({materialized['image_count']} files)"
    )
    if materialized["metadata_path"] is not None:
        print(f"Copied metadata file into Bronze metadata: {archive['metadata_target_path']}")

display(dbutils.fs.ls(bronze_paths["images"]))
display(dbutils.fs.ls(bronze_paths["metadata"]))


## Build Image Manifest

This step records the extracted Bronze image files and computes per-image checksums.

In [ ]:
image_manifest_rows = []
for archive in ARCHIVES:
    bronze_images_local_root = archive["bronze_images_local_root"]
    image_files = list_files_by_suffix(bronze_images_local_root, IMAGE_SUFFIXES)
    for image_path in image_files:
        target_path = f"{archive['bronze_images_path'].rstrip('/')}/{image_path.relative_to(bronze_images_local_root).as_posix()}"
        image_manifest_rows.append(
            {
                "image_id": image_path.stem,
                "source_split": archive["source_split"],
                "source_uri": target_path,
                "source_checksum": sha256_file(image_path),
            }
        )

print(f"Images in manifest: {len(image_manifest_rows)}")
display(spark.createDataFrame(image_manifest_rows).groupBy("source_split").count())


## Write Bronze Tables

This step loads the raw metadata file, aligns it with the copied images, merges the source rows into the Bronze table, and appends an ingestion run record.

In [ ]:
metadata_frames = [
    spark.read.option("header", True)
    .csv(archive["metadata_target_path"])
    .withColumn("source_split", F.lit(archive["source_split"]))
    .withColumn("label_source", F.lit(Path(archive["metadata_target_path"]).name))
    for archive in ARCHIVES
]
raw_metadata_df = metadata_frames[0]
for metadata_frame in metadata_frames[1:]:
    raw_metadata_df = raw_metadata_df.unionByName(metadata_frame, allowMissingColumns=True)

image_manifest_df = spark.createDataFrame(image_manifest_rows).dropDuplicates(["image_id", "source_split"])
source_metadata_df = (
    raw_metadata_df
    .withColumn("image_id", F.col("image").cast("string"))
    .join(image_manifest_df, on=["image_id", "source_split"], how="left")
    .select(
        F.col("image_id"),
        F.col("source_split"),
        F.col("source_uri"),
        F.col("patient_id").alias("patient_id"),
        F.col("lesion_id").alias("lesion_id"),
        F.col("dx").alias("label_raw"),
        F.col("label_source"),
        F.lit(None).cast("string").alias("label_normalized"),
        F.lit(None).cast("string").alias("acquisition_date"),
        F.col("source_checksum"),
    )
    .where(F.col("image_id").isNotNull())
)

ingestion_run_id = f"bronze-{uuid4().hex[:12]}"
started_at = datetime.now(timezone.utc)
source_metadata_df = (
    source_metadata_df
    .withColumn("ingestion_run_id", F.lit(ingestion_run_id))
    .withColumn("ingested_at", F.lit(started_at))
)

source_metadata_df.createOrReplaceTempView("staged_isic_2019_source_metadata")
spark.sql(
    f"""
    MERGE INTO {bronze_tables['source_metadata']} AS target
    USING staged_isic_2019_source_metadata AS source
    ON target.image_id = source.image_id AND target.source_split = source.source_split
    WHEN MATCHED THEN UPDATE SET
      target.source_split = source.source_split,
      target.source_uri = source.source_uri,
      target.patient_id = source.patient_id,
      target.lesion_id = source.lesion_id,
      target.label_raw = source.label_raw,
      target.label_source = source.label_source,
      target.label_normalized = source.label_normalized,
      target.acquisition_date = source.acquisition_date,
      target.source_checksum = source.source_checksum,
      target.ingestion_run_id = source.ingestion_run_id,
      target.ingested_at = source.ingested_at
    WHEN NOT MATCHED THEN INSERT *
    """
)

records_written = source_metadata_df.count()
finished_at = datetime.now(timezone.utc)
run_row = [{
    "ingestion_run_id": ingestion_run_id,
    "dataset_name": DATASET_CONFIG["dataset_name"],
    "source_version": DATASET_CONFIG["source_version"],
    "started_at": started_at,
    "finished_at": finished_at,
    "status": "success",
    "records_seen": records_written,
    "records_written": records_written,
}]
spark.createDataFrame(run_row).write.mode("append").saveAsTable(bronze_tables["ingestion_runs"])

print(f"Ingestion run id: {ingestion_run_id}")
print(f"Records written: {records_written}")


## Review Outputs

This final step shows the landed archive, Bronze file paths, and the most recent Bronze table contents for inspection.

In [ ]:
print("Landing archive contents:")
display(dbutils.fs.ls(landing_paths["dataset_archive"]))

print("Bronze metadata path contents:")
display(dbutils.fs.ls(bronze_paths["metadata"]))

print("Bronze images path sample:")
display(dbutils.fs.ls(bronze_paths["images"]))

print("Recent ingestion runs:")
display(spark.table(bronze_tables["ingestion_runs"]).orderBy("started_at", ascending=False))

print("Source metadata sample:")
display(spark.table(bronze_tables["source_metadata"]))
